# Task 2 - Recommendation System Development

This notebook develops recommendation-system analysis for the Commercial Director scenario, focusing on customer value, product performance and purchase behaviour.

## 1. Import the required libraries

Pandas and NumPy support data preparation. Matplotlib and Seaborn are included for later charts, and Scikit-Learn's cosine similarity will support a collaborative recommendation approach.

In [23]:
# Import libraries for tables, calculations and charts.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import cosine similarity for a later collaborative-filtering recommender.
from sklearn.metrics.pairwise import cosine_similarity

## 2. Load the allocated datasets

Load the assigned customer, product and transaction CSV files from the `Datasets` folder. Keeping these as separate DataFrames makes each source easier to inspect before joining.

In [24]:
# Set paths to the three allocated CSV files.
customers_path = "../Datasets/Customers_members.csv"
products_path = "../Datasets/Products_overview.csv"
transactions_path = "../Datasets/Transactions_history.csv"

try:
    # Load each source dataset into its own DataFrame.
    customers_raw = pd.read_csv(customers_path)
    products_raw = pd.read_csv(products_path)
    transactions_raw = pd.read_csv(transactions_path)
    print("Customer, product and transaction datasets loaded successfully.")
except FileNotFoundError as error:
    print(f"A dataset file could not be found: {error.filename}")
    print("Check the CSV filenames and confirm that the notebook is in the Ca1-folder directory.")

Customer, product and transaction datasets loaded successfully.


## 3. Inspect the source datasets

Check each dataset's dimensions, sample rows, column names and data types before combining them. This helps confirm the join keys and the meaning of the transaction measures.

In [25]:
# Display dataset dimensions and the first five rows for each source.
for dataset_name, dataset in {
    "Customers": customers_raw,
    "Products": products_raw,
    "Transactions": transactions_raw
}.items():
    print(f"{dataset_name} dataset shape: {dataset.shape}")
    display(dataset.head())

# Inspect column names, data types and non-null counts.
print("Customers dataset information:")
customers_raw.info()

print("Products dataset information:")
products_raw.info()

print("Transactions dataset information:")
transactions_raw.info()

Customers dataset shape: (1000, 5)


,CustomerID,AgeGroup,Region,LoyaltyLevel,MembershipYears
0,1,36-50,East,Bronze,9
1,2,26-35,East,Bronze,4
2,3,18-25,East,Silver,10
3,4,36-50,East,Bronze,14
4,5,26-35,North,Bronze,7


Products dataset shape: (200, 6)


,ProductID,Category,Brand,Price,Rating,PopularityScore
0,1,Electronics,TechMax,474.45,4.54,3
1,2,Gaming,SmartBuy,285.11,3.16,2
2,3,books,HomePro,370.79,3.01,1
3,4,Home,TechMax,416.38,3.38,1
4,5,Books,TechMax,62.37,3.42,1


Transactions dataset shape: (19166, 5)


,CustomerID,ProductID,PurchaseCount,LastPurchaseDays,Spend
0,1,82,3,162,104.01
1,2,7,4,276,387.48
2,3,170,4,215,1507.40
3,3,9,1,251,341.40
4,4,64,3,354,1250.40


Customers dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   CustomerID       1000 non-null   int64
 1   AgeGroup         1000 non-null   str  
 2   Region           1000 non-null   str  
 3   LoyaltyLevel     1000 non-null   str  
 4   MembershipYears  1000 non-null   int64
dtypes: int64(2), str(3)
memory usage: 39.2 KB
Products dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ProductID        200 non-null    int64  
 1   Category         200 non-null    str    
 2   Brand            200 non-null    str    
 3   Price            200 non-null    float64
 4   Rating           185 non-null    float64
 5   PopularityScore  200 non-null    int64  
dtypes: float64(2), int64(2),

## 4. Assess data quality and key integrity

Summarise missing values, exact duplicate rows and duplicate entity keys. Transaction duplicates are reported for review rather than removed automatically because the file has no transaction ID to distinguish accidental duplicates from repeated purchases.

In [26]:
# Summarise missing cells and fully duplicated rows in each source dataset.
data_quality_summary = pd.DataFrame({
    "Missing Cells": [
        customers_raw.isnull().sum().sum(),
        products_raw.isnull().sum().sum(),
        transactions_raw.isnull().sum().sum()
    ],
    "Exact Duplicate Rows": [
        customers_raw.duplicated().sum(),
        products_raw.duplicated().sum(),
        transactions_raw.duplicated().sum()
    ]
}, index=["Customers", "Products", "Transactions"])

# Show missing counts by field to identify which columns need attention.
missing_by_column = pd.concat({
    "Customers": customers_raw.isnull().sum(),
    "Products": products_raw.isnull().sum(),
    "Transactions": transactions_raw.isnull().sum()
}, names=["Dataset", "Column"]).to_frame(name="Missing Values")

missing_by_column["Missing Percentage"] = pd.concat({
    "Customers": customers_raw.isnull().mean() * 100,
    "Products": products_raw.isnull().mean() * 100,
    "Transactions": transactions_raw.isnull().mean() * 100
}, names=["Dataset", "Column"])

print("Dataset-level quality summary:")
display(data_quality_summary)

print("Missing values by column:")
display(missing_by_column)

Dataset-level quality summary:


,Missing Cells,Exact Duplicate Rows
Customers,0,0
Products,15,0
Transactions,0,384


Missing values by column:


Missing Values  Missing Percentage
Dataset      Column                                              
Customers    CustomerID                     0                 0.0
             AgeGroup                       0                 0.0
             Region                         0                 0.0
             LoyaltyLevel                   0                 0.0
             MembershipYears                0                 0.0
Products     ProductID                      0                 0.0
             Category                       0                 0.0
             Brand                          0                 0.0
             Price                          0                 0.0
             Rating                        15                 7.5
             PopularityScore                0                 0.0
Transactions CustomerID                     0                 0.0
             ProductID                      0                 0.0
             PurchaseCount                  0                 0.0
             LastPurchaseDays               0                 0.0
             Spend                          0                 0.0

In [27]:
# Check whether the entity keys are unique in their respective datasets.
key_duplicate_summary = pd.Series({
    "Duplicate CustomerID values": customers_raw["CustomerID"].duplicated().sum(),
    "Duplicate ProductID values": products_raw["ProductID"].duplicated().sum()
}, name="Duplicate key count")

print("Duplicate entity keys:")
display(key_duplicate_summary)

# Check that each transaction references an existing customer and product.
unmatched_customer_count = (~transactions_raw["CustomerID"].isin(customers_raw["CustomerID"])).sum()
unmatched_product_count = (~transactions_raw["ProductID"].isin(products_raw["ProductID"])).sum()

print(f"Transactions with an unknown CustomerID: {unmatched_customer_count}")
print(f"Transactions with an unknown ProductID: {unmatched_product_count}")

Duplicate entity keys:


Duplicate CustomerID values    0
Duplicate ProductID values     0
Name: Duplicate key count, dtype: int64

Transactions with an unknown CustomerID: 0
Transactions with an unknown ProductID: 0


## 5. Integrate the datasets

Create clean working copies, standardize product category capitalization, then enrich transaction records with customer and product details. The join validation checks that each transaction links to one customer and one product.

In [28]:
# Create working copies and standardize category labels for consistent grouping.
customers = customers_raw.copy()
products = products_raw.copy()
transactions = transactions_raw.copy()

products["Category"] = products["Category"].str.strip().str.title()

# Add customer and product details to each transaction.
transactions_enriched = transactions.merge(
    customers,
    on="CustomerID",
    how="left",
    validate="many_to_one"
).merge(
    products,
    on="ProductID",
    how="left",
    validate="many_to_one"
)

print(f"Transactions before integration: {len(transactions)}")
print(f"Transactions after integration: {len(transactions_enriched)}")
print(f"Integrated dataset shape: {transactions_enriched.shape}")
display(transactions_enriched.head())

Transactions before integration: 19166
Transactions after integration: 19166
Integrated dataset shape: (19166, 14)


,CustomerID,ProductID,PurchaseCount,LastPurchaseDays,Spend,AgeGroup,Region,LoyaltyLevel,MembershipYears,Category,Brand,Price,Rating,PopularityScore
0,1,82,3,162,104.01,36-50,East,Bronze,9,Books,HomePro,34.67,3.56,5
1,2,7,4,276,387.48,26-35,East,Bronze,4,Home,SmartBuy,96.87,3.79,49
2,3,170,4,215,1507.40,18-25,East,Silver,10,Kitchen,FitLife,376.85,2.78,4
3,3,9,1,251,341.40,18-25,East,Silver,10,Gaming,SmartBuy,341.40,4.23,3
4,4,64,3,354,1250.40,36-50,East,Bronze,14,Gaming,TechMax,416.80,NaN,1


## 6. Create the customer-product interaction matrix

The matrix aggregates `PurchaseCount` for each customer-product pair. A zero means no recorded purchase interaction in the transaction data. A second matrix sums `Spend` for later customer-value analysis.

In [29]:
# Aggregate repeated customer-product records into total purchase counts.
interaction_matrix = transactions.pivot_table(
    index="CustomerID",
    columns="ProductID",
    values="PurchaseCount",
    aggfunc="sum",
    fill_value=0
)

# Retain a spend matrix for customer and product value analysis.
spend_matrix = transactions.pivot_table(
    index="CustomerID",
    columns="ProductID",
    values="Spend",
    aggfunc="sum",
    fill_value=0
)

print(f"Interaction matrix shape (customers, products): {interaction_matrix.shape}")
print(f"Number of recorded customer-product interactions: {(interaction_matrix > 0).sum().sum()}")
display(interaction_matrix.head())

Interaction matrix shape (customers, products): (1000, 200)
Number of recorded customer-product interactions: 15692


ProductID,1,2,3,4,5,6,7,8,9,10,...,191,192,193,194,195,196,197,198,199,200
CustomerID,,,,,,,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,4,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 7. Customer Analytics

This section summarizes customer spending and purchase frequency for the Commercial Director scenario. The transaction file contains 384 exact duplicate rows and no transaction ID, so the rankings below use all supplied rows and should be treated as provisional until those repeats are confirmed.

### 7.1 Build customer purchase summaries

Aggregate each customer's spend and purchase count, and add product breadth and profile details. Here, purchase frequency means the sum of `PurchaseCount`, not the number of dataset rows.

In [30]:
# Aggregate transaction measures for each customer.
customer_purchase_summary = (
    transactions_enriched.groupby("CustomerID")
    .agg(
        TotalSpend=("Spend", "sum"),
        TotalPurchases=("PurchaseCount", "sum"),
        TransactionRows=("ProductID", "size"),
        UniqueProducts=("ProductID", "nunique"),
        MostRecentPurchaseDays=("LastPurchaseDays", "min")
    )
    .reset_index()
)

# Add customer profile details for segment comparisons.
customer_purchase_summary = customer_purchase_summary.merge(
    customers[["CustomerID", "AgeGroup", "Region", "LoyaltyLevel", "MembershipYears"]],
    on="CustomerID",
    how="left",
    validate="one_to_one"
)

# Calculate average spend per recorded purchase unit.
customer_purchase_summary["AverageSpendPerPurchase"] = (
    customer_purchase_summary["TotalSpend"]
    / customer_purchase_summary["TotalPurchases"].replace(0, np.nan)
)

customer_purchase_summary.head()

,CustomerID,TotalSpend,TotalPurchases,TransactionRows,UniqueProducts,MostRecentPurchaseDays,AgeGroup,Region,LoyaltyLevel,MembershipYears,AverageSpendPerPurchase
0,1,104.01,3,1,1,162,36-50,East,Bronze,9,34.67
1,2,387.48,4,1,1,276,26-35,East,Bronze,4,96.87
2,3,1848.80,5,2,2,215,18-25,East,Silver,10,369.76
3,4,3469.52,8,3,2,242,36-50,East,Bronze,14,433.69
4,5,331.78,2,1,1,23,26-35,North,Bronze,7,165.89


### 7.2 Top 10 customers by spend and purchase frequency

These rankings show the customers with the highest recorded total spend and purchase counts. Keeping the measures separate avoids treating high spend and frequent purchasing as the same behaviour.

In [31]:
# Select columns that make the customer rankings easy to interpret.
customer_ranking_columns = [
    "CustomerID",
    "Region",
    "LoyaltyLevel",
    "TotalSpend",
    "TotalPurchases",
    "UniqueProducts"
]

# Rank customers by total spend.
top_10_customers_by_spend = (
    customer_purchase_summary.nlargest(10, "TotalSpend")
    [customer_ranking_columns]
    .reset_index(drop=True)
)

# Rank customers by total purchase count.
top_10_customers_by_frequency = (
    customer_purchase_summary.nlargest(10, "TotalPurchases")
    [customer_ranking_columns]
    .reset_index(drop=True)
)

print("Top 10 customers by total recorded spend:")
display(top_10_customers_by_spend)

print("Top 10 customers by total purchase frequency:")
display(top_10_customers_by_frequency)

Top 10 customers by total recorded spend:


,CustomerID,Region,LoyaltyLevel,TotalSpend,TotalPurchases,UniqueProducts
0,395,East,Bronze,27323.95,86,23
1,751,West,Gold,23084.09,72,19
2,550,East,Silver,23065.13,87,25
3,999,East,Silver,23012.88,72,19
4,298,West,Bronze,22883.98,76,22
5,454,East,Silver,22750.97,76,20
6,57,West,Bronze,22713.06,71,24
7,355,South,Gold,22533.65,79,22
8,546,North,Silver,22399.85,79,24
9,676,North,Silver,22273.50,71,23


Top 10 customers by total purchase frequency:


,CustomerID,Region,LoyaltyLevel,TotalSpend,TotalPurchases,UniqueProducts
0,550,East,Silver,23065.13,87,25
1,395,East,Bronze,27323.95,86,23
2,664,West,Bronze,20840.02,84,22
3,290,North,Bronze,22138.19,83,27
4,831,West,Bronze,19476.28,83,26
5,812,West,Gold,20100.33,81,22
6,879,East,Bronze,21698.92,81,24
7,289,West,Silver,20071.17,80,20
8,480,West,Bronze,20891.12,80,24
9,571,South,Bronze,17691.76,80,19


### 7.3 Customer value analysis

Use total-spend percentiles to describe high-, medium- and low-spend customer groups. Compare average spend, purchase count, product breadth and days since most recent purchase, plus the high-spend group's loyalty and regional mix.

In [32]:
# Assign descriptive spend tiers using each customer's percentile rank.
spend_percentile = customer_purchase_summary["TotalSpend"].rank(pct=True)
customer_purchase_summary["CustomerValueTier"] = np.select(
    [spend_percentile >= 0.75, spend_percentile >= 0.25],
    ["High", "Medium"],
    default="Low"
)

# Compare purchasing behaviour across spend-based value tiers.
customer_value_analysis = (
    customer_purchase_summary.groupby("CustomerValueTier")
    .agg(
        CustomerCount=("CustomerID", "nunique"),
        AverageTotalSpend=("TotalSpend", "mean"),
        AveragePurchaseCount=("TotalPurchases", "mean"),
        AverageUniqueProducts=("UniqueProducts", "mean"),
        AverageDaysSinceLastPurchase=("MostRecentPurchaseDays", "mean")
    )
    .reindex(["High", "Medium", "Low"])
)

# Profile high-spend customers by loyalty level and region.
high_value_customer_profile = (
    customer_purchase_summary[customer_purchase_summary["CustomerValueTier"] == "High"]
    .groupby(["Region", "LoyaltyLevel"])
    .agg(
        CustomerCount=("CustomerID", "nunique"),
        AverageTotalSpend=("TotalSpend", "mean"),
        AveragePurchaseCount=("TotalPurchases", "mean")
    )
    .sort_values("AverageTotalSpend", ascending=False)
)

print("Customer value summary by spend tier:")
display(customer_value_analysis)

print("High-spend customer profile by region and loyalty level:")
display(high_value_customer_profile.head(10))

Customer value summary by spend tier:


,CustomerCount,AverageTotalSpend,AveragePurchaseCount,AverageUniqueProducts,AverageDaysSinceLastPurchase
CustomerValueTier,,,,,
High,251,17953.555777,67.390438,20.561753,14.039841
Medium,500,11695.504480,49.276000,16.356000,19.282000
Low,249,5533.945382,25.554217,9.449799,50.473896


High-spend customer profile by region and loyalty level:


,,CustomerCount,AverageTotalSpend,AveragePurchaseCount
Region,LoyaltyLevel,,,
East,Bronze,29,18322.112759,68.620690
West,Gold,14,18303.992857,67.500000
East,Silver,21,18282.746190,69.190476
North,Silver,17,18258.088824,66.941176
West,Silver,16,18104.221250,69.000000
North,Bronze,36,18058.345000,68.555556
East,Gold,9,17970.122222,62.333333
South,Bronze,27,17785.996296,68.740741
West,Bronze,41,17768.066341,66.219512


## 8. Product Analytics

This section evaluates product performance using transaction `Spend` as recorded revenue and `PurchaseCount` as purchase volume. Product revenue totals include all supplied transaction rows, including the 384 exact duplicates flagged in the data-quality check, so treat totals as provisional until those repeats are confirmed.

### 8.1 Product-level performance

Aggregate revenue and purchase counts by product, and retain ratings and popularity scores as context for product performance.

In [33]:
# Summarise each product's observed revenue, purchase count and available product attributes.
product_performance = (
    transactions_enriched.groupby(
        ["ProductID", "Category", "Brand"],
        dropna=False
    )
    .agg(
        TotalRevenue=("Spend", "sum"),
        TotalPurchaseCount=("PurchaseCount", "sum"),
        TransactionRows=("CustomerID", "size"),
        AverageRating=("Rating", "mean"),
        AverageListedPrice=("Price", "mean"),
        AveragePopularityScore=("PopularityScore", "mean")
    )
    .reset_index()
    .sort_values("TotalRevenue", ascending=False)
)

product_performance.head()

,ProductID,Category,Brand,TotalRevenue,TotalPurchaseCount,TransactionRows,AverageRating,AverageListedPrice,AveragePopularityScore
101,102,Books,FitLife,1756773.72,3627,1424,2.88,484.36,38.0
6,7,Home,SmartBuy,488127.93,5039,1994,3.79,96.87,49.0
40,41,Home,HomePro,446687.43,1097,454,NaN,407.19,12.0
133,134,Kitchen,SmartBuy,415775.14,1397,553,3.44,297.62,14.0
7,8,Home,HomePro,240130.74,703,270,3.71,341.58,6.0


### 8.2 Top 10 products by revenue

Rank products by recorded spend and display purchase volume and average rating to provide additional context beyond revenue alone.

In [34]:
# Display the ten products with the highest recorded revenue.
top_10_products_by_revenue = (
    product_performance[
        [
            "ProductID",
            "Category",
            "Brand",
            "TotalRevenue",
            "TotalPurchaseCount",
            "AverageRating"
        ]
    ]
    .head(10)
    .reset_index(drop=True)
)

top_10_products_by_revenue

,ProductID,Category,Brand,TotalRevenue,TotalPurchaseCount,AverageRating
0,102,Books,FitLife,1756773.72,3627,2.88
1,7,Home,SmartBuy,488127.93,5039,3.79
2,41,Home,HomePro,446687.43,1097,NaN
3,134,Kitchen,SmartBuy,415775.14,1397,3.44
4,8,Home,HomePro,240130.74,703,3.71
5,184,Fitness,SmartBuy,195388.13,973,4.36
6,45,Kitchen,NextGen,190771.02,418,3.86
7,65,Kitchen,TechMax,187267.92,498,3.19
8,81,Home,FitLife,182543.45,445,4.38
9,137,Electronics,NextGen,172986.45,377,3.06


### 8.3 Revenue by category and brand

These summaries compare category and brand revenue, purchase counts and each group's share of observed spend.

In [35]:
# Group product performance by category.
revenue_by_category = (
    transactions_enriched.groupby("Category", dropna=False)
    .agg(
        TotalRevenue=("Spend", "sum"),
        TotalPurchaseCount=("PurchaseCount", "sum"),
        Products=("ProductID", "nunique")
    )
    .sort_values("TotalRevenue", ascending=False)
)
revenue_by_category["RevenueSharePercent"] = (
    revenue_by_category["TotalRevenue"]
    / revenue_by_category["TotalRevenue"].sum()
    * 100
)

# Group product performance by brand.
revenue_by_brand = (
    transactions_enriched.groupby("Brand", dropna=False)
    .agg(
        TotalRevenue=("Spend", "sum"),
        TotalPurchaseCount=("PurchaseCount", "sum"),
        Products=("ProductID", "nunique")
    )
    .sort_values("TotalRevenue", ascending=False)
)
revenue_by_brand["RevenueSharePercent"] = (
    revenue_by_brand["TotalRevenue"]
    / revenue_by_brand["TotalRevenue"].sum()
    * 100
)

print("Revenue by product category:")
display(revenue_by_category)

print("Revenue by brand:")
display(revenue_by_brand)

# Summarise the leading category and brand by recorded revenue.
print(f"Highest-revenue category: {revenue_by_category.index[0]}")
print(f"Highest-revenue brand: {revenue_by_brand.index[0]}")

Revenue by product category:


,TotalRevenue,TotalPurchaseCount,Products,RevenueSharePercent
Category,,,,
Books,2795746.40,8933,37,23.829996
Home,2337945.87,11719,30,19.927859
Kitchen,1808430.27,6584,26,15.414448
Electronics,1796995.49,8811,38,15.316981
Fitness,1581412.97,6327,34,13.479429
Gaming,1411516.14,5542,35,12.031286


Revenue by brand:


,TotalRevenue,TotalPurchaseCount,Products,RevenueSharePercent
Brand,,,,
FitLife,3348546.16,9829,33,28.541874
SmartBuy,2632371.13,14441,41,22.437441
HomePro,1911437.37,7894,33,16.292445
TechMax,1620948.69,5768,35,13.816418
NextGen,1306117.89,4748,29,11.132907
PrimeGoods,912625.90,5236,29,7.778914


Highest-revenue category: Books
Highest-revenue brand: FitLife
